# TMJDLC — Javtiful Colab Download Smoke Test

Run top to bottom to test stream resolution, yt-dlp download, FFprobe validation, and four FFmpeg screenshots. Temporary stream URLs and headers stay in runtime memory; they are never printed or persisted.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!python -m pip install -q --upgrade pip
!python -m pip install -q "playwright>=1.50" "yt-dlp>=2025.1" "requests>=2.32" "beautifulsoup4>=4.12"
!playwright install --with-deps chromium

In [ ]:
import pathlib, shutil, subprocess, json, os, sys, time
WORKDIR = pathlib.Path('/content/tmjdlc-test')
REPO_DIR = WORKDIR / 'TMJDLC'
WORKDIR.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
REPO_URL = 'https://github.com/Telebotfaroff/TMJDLC.git'
BRANCH = 'main'
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO_DIR)], check=True)
%cd /content/tmjdlc-test/TMJDLC
!python -m pip install -q -e .

## Configure the test

Default URL is the post already confirmed by the resolver workflow. Change it when testing another post.

In [ ]:
POST_URL = input('Javtiful post URL [https://javtiful.com/video/113371/dvmm-444]: ').strip() or 'https://javtiful.com/video/113371/dvmm-444'
QUALITY = input('Preferred quality (360/480/720/1080) [720]: ').strip() or '720'
if QUALITY not in {'360','480','720','1080','1440','2160'}: raise ValueError('Unsupported quality')
TIMEOUT, CAPTURE_SECONDS = 45, 20
OUTPUT_DIR = WORKDIR / 'output'
SCREENSHOT_DIR = OUTPUT_DIR / 'screenshots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
print('Post:', POST_URL, '| quality:', QUALITY + 'p')

## Resolve the stream

The resolver returns a temporary URL and session headers. This cell deliberately hides both.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tmjdlc.providers.javtiful import JavtifulStreamDetector

def field(obj, name, default=None):
    return obj.get(name, default) if isinstance(obj, dict) else getattr(obj, name, default)

# Colab/IPython runs an asyncio event loop. Playwright's Sync API must run
# in a separate worker thread to avoid the 'asyncio loop is running' error.
def resolve_stream():
    detector = JavtifulStreamDetector(
        timeout_ms=TIMEOUT * 1000,
        capture_ms=CAPTURE_SECONDS * 1000,
    )
    return detector.detect(POST_URL, preferred_quality=int(QUALITY))

with ThreadPoolExecutor(max_workers=1) as pool:
    stream_result = pool.submit(resolve_stream).result()

stream_url = field(stream_result, 'stream_url')
stream_format = field(stream_result, 'format', 'unknown')
selected_quality = field(stream_result, 'quality', QUALITY)
duration_seconds = field(stream_result, 'duration_seconds')
headers = field(stream_result, 'headers', {}) or {}
if not stream_url:
    raise RuntimeError('Resolver returned no stream URL')

print('Resolved:', stream_format, '| quality:', str(selected_quality) + 'p', '| duration:', round(duration_seconds, 2) if duration_seconds else 'unknown')
print('Session headers captured:', bool(headers), '| signed URL hidden')


## Download with yt-dlp

Downloads and merges the HLS stream into a local media file. The signed URL is not displayed.

In [ ]:
import yt_dlp
video_path = OUTPUT_DIR / 'javtiful_video.mp4'
opts = {
 'outtmpl': str(OUTPUT_DIR / 'javtiful_video.%(ext)s'), 'format': 'best',
 'merge_output_format': 'mp4', 'noplaylist': True, 'retries': 10,
 'fragment_retries': 10, 'skip_unavailable_fragments': False,
 'concurrent_fragment_downloads': 4, 'http_headers': dict(headers),
 'quiet': True, 'no_warnings': False,
 'progress_hooks': [lambda d: print('Download:', d.get('status'), d.get('_percent_str','').strip(), d.get('_speed_str','').strip(), 'ETA', d.get('_eta_str','').strip()) if d.get('status') == 'downloading' else None]
}
with yt_dlp.YoutubeDL(opts) as ydl: ydl.download([stream_url])
candidates = [p for p in OUTPUT_DIR.glob('javtiful_video.*') if p.is_file() and p.suffix.lower() not in {'.part','.ytdl','.json'}]
if not candidates: raise FileNotFoundError('No downloaded media file found')
video_path = max(candidates, key=lambda p:p.stat().st_size)
print('Downloaded:', video_path.name, '| MiB:', round(video_path.stat().st_size/1048576,2))

## Verify with FFprobe

In [ ]:
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size,format_name:stream=codec_type,codec_name,width,height','-of','json',str(video_path)],capture_output=True,text=True)
if probe.returncode: raise RuntimeError('FFprobe failed: '+probe.stderr[-1200:])
probe_data = json.loads(probe.stdout)
print(json.dumps(probe_data, indent=2))

## Generate four screenshots with FFmpeg

In [ ]:
try: duration = float(probe_data.get('format',{}).get('duration'))
except (TypeError,ValueError): duration = None
timestamps = [max(1,int(duration*f)) for f in (.15,.35,.60,.85)] if duration and duration > 8 else [10,30,60,90]
generated = []
for i, seconds in enumerate(timestamps,1):
    out = SCREENSHOT_DIR / f'frame_{i}.jpg'
    result = subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-ss',str(seconds),'-i',str(video_path),'-frames:v','1','-q:v','2',str(out)],capture_output=True,text=True)
    if result.returncode == 0 and out.exists() and out.stat().st_size: generated.append(out)
    else: print('Frame failed at', seconds, 'seconds:', result.stderr[-400:])
print('Screenshots generated:', len(generated))
if not generated: raise RuntimeError('No screenshots generated')

## Preview screenshots

In [ ]:
from IPython.display import display, Image
for p in generated:
    print(p.name)
    display(Image(filename=str(p)))

## Download test outputs

Creates one ZIP containing the MP4 and generated screenshots.

In [ ]:
import zipfile
from google.colab import files
zip_path = WORKDIR / 'tmjdlc-smoke-test-output.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(video_path, arcname=video_path.name)
    for p in generated: z.write(p, arcname='screenshots/'+p.name)
print('Archive size MiB:', round(zip_path.stat().st_size/1048576,2))
files.download(str(zip_path))

## Optional cleanup

Run only after downloading the ZIP.

In [ ]:
# Uncomment to remove temporary files:
# shutil.rmtree(WORKDIR, ignore_errors=True)